#**[Using AI to Detect Proper Exercise Form](https://www.sciencebuddies.org/science-fair-projects/project-ideas/ArtificialIntelligence_p027/artificial-intelligence/exercise_form)**

This notebook was developed by Science Buddies [www.sciencebuddies.org](https://www.sciencebuddies.org/) as part of a science project to allow students to explore and learn about artificial intelligence. For personal use, this notebook can be downloaded and modified with attribution. For all other uses, please see our [Terms and Conditions of Fair Use](https://www.sciencebuddies.org/about/terms-and-conditions-of-fair-use).  

**Troubleshooting tips**
*   Read the written instructions at Science Buddies and the text and comments on this page carefully.
*   If you make changes that break the code, you can download a fresh copy of this notebook and start over.

*   If you are using this notebook for a science project and need help, visit our [Ask an Expert](https://www.sciencebuddies.org/science-fair-projects/ask-an-expert-intro) forum for assistance.

## **How To Use This Notebook**

This notebook contains text fields, like this one, that give you information about the project and instructions.

In [ ]:
# There are also code blocks, like this one.

# The green text in a code block are comments. Comments are descriptions of what the code does.

# The non-green text in a code block is the Python code. Click on the triangle in the top left corner to run this code block.

print("Congratulations, you ran a code block! Try changing the text in the code and running it again.")

# Importing Libraries

In [ ]:
# Install the Ultralytics library, which includes the YOLO (You Only Look Once) models and tools for object detection and pose estimation
!pip install ultralytics

# Install OpenCV, a computer vision library used for image and video processing
!pip install opencv-python

In [ ]:
# Deep Learning / Object Detection
from ultralytics import YOLO  # YOLO model for object detection and pose estimation

# Computer Vision
import cv2  # OpenCV for image and video processing

# Data Handling
import pandas as pd  # For working with dataframes (structured data)
import numpy as np   # For numerical computations and array operations
import os            # For interacting with the operating system (e.g., file paths)
from glob import glob  # For finding file paths matching a pattern
from glob import iglob # For finding file paths matching a pattern

# Machine Learning
import joblib        # For saving and loading machine learning models

# Machine Learning
from xgboost import XGBClassifier  # Gradient boosting classifier
from sklearn.model_selection import train_test_split  # To split data into training and test sets
from sklearn.metrics import classification_report      # To evaluate classification model performance

In [ ]:
from google.colab import drive  # Import the drive module to access Google Drive from Colab

# Mount Google Drive to the Colab environment at the specified path
# `force_remount=True` ensures the drive will be re-mounted even if it's already mounted
drive.mount('/content/drive', force_remount=True)

In [ ]:
# Define the base directory where all data is stored
base_dir = "/content/drive/MyDrive/exercise_form"

# Define paths
paths = {
    "good_form_videos": os.path.join(base_dir, "exercise_videos/good_form"),
    "bad_form_videos": os.path.join(base_dir, "exercise_videos/bad_form"),
    "good_form_keypoints": os.path.join(base_dir, "exercise_keypoints/good_form"),
    "bad_form_keypoints": os.path.join(base_dir, "exercise_keypoints/bad_form"),
    "good_form_labeled": os.path.join(base_dir, "exercise_labeled_videos/good_form"),
    "bad_form_labeled": os.path.join(base_dir, "exercise_labeled_videos/bad_form"),
    "test_videos": os.path.join(base_dir, "test_videos"),
}

# Create directories if they don't exist
for path_name, path in paths.items():
    os.makedirs(path, exist_ok=True)  # Create the directory if it doesn't exist
    print(f"Checked/created: {path_name} -> {path}")

good_form_path = paths["good_form_videos"]
bad_form_path = paths["bad_form_videos"]
good_form_keypoints_path = paths["good_form_keypoints"]
bad_form_keypoints_path = paths["bad_form_keypoints"]
good_form_labeled_video_path = paths["good_form_labeled"]
bad_form_labeled_video_path = paths["bad_form_labeled"]
test_videos_path = paths["test_videos"]

# 1. Saving KeyPoints

Code Block 1A

In [ ]:
# --- Load YOLO Pose Model ---
model = YOLO("yolov8n-pose.pt")  # Load the YOLOv8 nano pose model (alternatives: yolov8s/m for better accuracy)

# Loop through good and bad form folders with label 0 (good) and 1 (bad)
for label, folder in enumerate([good_form_path, bad_form_path]):
    video_files = []
    for ext in ["*.mp4", "*.MP4", "*.Mp4", "*.mP4"]:
        video_files.extend(iglob(os.path.join(folder, ext)))
    # Get all .MP4 files in the current folder
    for video_path in video_files:
        short_video_path = os.path.basename(video_path)  # Extract just the video filename

        # Define output paths for labeled video and keypoints CSV
        if folder == good_form_path:
            output_video_path = os.path.join(good_form_labeled_video_path, "output_" + short_video_path)
            output_csv = os.path.join(good_form_keypoints_path, short_video_path.replace(".MP4", "_keypoints.csv"))
        elif folder == bad_form_path:
            output_video_path = os.path.join(bad_form_labeled_video_path, "output_" + short_video_path)
            output_csv = os.path.join(bad_form_keypoints_path, short_video_path.replace(".MP4", "_keypoints.csv"))

        # --- Initialize Video Capture ---
        cap = cv2.VideoCapture(video_path)  # Open the video file
        frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))   # Get video width
        frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) # Get video height
        fps = cap.get(cv2.CAP_PROP_FPS)                        # Get frames per second

        # --- Initialize Video Writer for Annotated Output ---
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Define the codec
        out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))  # Set up the writer

        frame_num = 0  # Frame counter
        data = []      # List to store keypoints data for CSV
        frame_skip = 1  # Set frame interval (1 = every frame, 2 = every other frame, etc.)

        # --- Process Each Frame in the Video ---
        while cap.isOpened():
            ret, frame = cap.read()  # Read next frame
            if not ret:
                break  # Exit loop if no more frames

            # Only process selected frames (e.g., every 1st, 3rd, etc.)
            if frame_num % frame_skip == 0:
                results = model(frame, verbose=False)  # Run YOLO pose estimation

                # Draw keypoints and skeletons on the frame
                annotated_frame = results[0].plot()

                keypoints = results[0].keypoints  # Get keypoint data from results
                if keypoints is not None and len(keypoints.xy) > 0:
                    for i, kp in enumerate(keypoints.xy[0]):  # Loop through keypoints for the first detected person
                        x, y = kp[0].item(), kp[1].item()        # Extract x, y coordinates
                        conf = keypoints.conf[0][i].item()       # Confidence score
                        data.append({
                            'frame': frame_num,
                            'keypoint_id': i,
                            'x': x,
                            'y': y,
                            'confidence': conf
                        })

                # Write the annotated frame to the output video
                out.write(annotated_frame)

            frame_num += 1  # Increment frame count

        # --- Finalize Video and Save Keypoints ---
        cap.release()  # Release video capture
        out.release()  # Release video writer

        # Save keypoints data to a CSV file
        df = pd.DataFrame(data)
        df.to_csv(output_csv, index=False)

        # Print confirmation messages
        print(f"Keypoints saved to {output_csv}")
        print(f"Video saved to {output_video_path}")
        print("")

# 2. Training the Model

Code Block 2A

In [ ]:
# --- Helper Function to Extract Features from a Keypoints CSV File ---
def extract_features_from_csv(csv_path):
    # Load keypoints CSV into a DataFrame
    df = pd.read_csv(csv_path)

    # Pivot the data: rows = frames, columns = (x, y) coordinates for each keypoint_id
    grouped = df.groupby(['frame', 'keypoint_id'])[['x', 'y']].mean().unstack()

    # --- Feature Extraction ---
    # Use statistical summaries:
    features = grouped.mean().tolist()  # Mean position of each keypoint (avg posture)
    features += grouped.std().tolist()  # Standard deviation (movement/instability)

    return features

# --- Load All Keypoint Data and Labels ---
X = []  # Feature vectors for each video
y = []  # Corresponding labels: 0 = good form, 1 = bad form

# Loop through both good and bad form keypoints folders
for label, folder in enumerate([good_form_keypoints_path, bad_form_keypoints_path]):
    for path in glob(f"{folder}/*.csv"):
        features = extract_features_from_csv(path)  # Extract features from each CSV file
        if features:  # Only include if valid features were extracted
            X.append(features)  # Add to feature set
            y.append(label)     # Add corresponding label

# Convert to arrays
X = np.array(X)
y = np.array(y)

Code Block 2B

In [ ]:
# Split the dataset into training and testing sets
# 80% of the data for training, 20% for testing
# random_state=42 ensures reproducibility
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and Train the XGBoost Classifier
model = XGBClassifier()       # Create an instance of the XGBoost classifier
model.fit(X_train, y_train)   # Train the model on the training data

# Evaluate the Model on the Test Set
y_pred = model.predict(X_test)  # Predict labels for the test data

# Print a classification report including precision, recall, f1-score, and accuracy
print(classification_report(y_test, y_pred, target_names=["Good Form", "Bad Form"]))

Code Block 2C

In [ ]:
# Define the path
model_path = '/content/drive/My Drive/exercise_form/form_classifier_model.pkl'

# Make sure the directory exists
os.makedirs(os.path.dirname(model_path), exist_ok=True)

# Save the model
joblib.dump(model, model_path)

# 3. Testing the Model

Code Block 3A

In [ ]:
# # TODO: Edit video name
video_name = "bad_form_test.MP4"
video_path = base_dir + "/test_videos/" + video_name
test_videos = base_dir + "/test_videos"

output_video_path = os.path.join(test_videos,"output_" + video_name)
output_csv = os.path.join(test_videos, video_name.replace(".MP4", "_keypoints.csv"))

# Save keypoints
model = YOLO("yolov8n-pose.pt") # or yolov8s/pose, yolov8m/pose for better accuracy

# --- Initialize Video ---
cap = cv2.VideoCapture(video_path)
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

# --- Initialize Video Writer ---
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

frame_num = 0
data = []
frame_skip = 1  # process every 3rd frame to speed things up

# --- Process Video ---
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    if frame_num % frame_skip == 0:
        results = model(frame, verbose=False)

        # Draw keypoints and skeletons directly on the frame
        annotated_frame = results[0].plot()

        keypoints = results[0].keypoints
        if keypoints is not None and len(keypoints.xy) > 0:
            for i, kp in enumerate(keypoints.xy[0]):  # first person only
                x, y = kp[0].item(), kp[1].item()
                conf = keypoints.conf[0][i].item()
                data.append({
                    'frame': frame_num,
                    'keypoint_id': i,
                    'x': x,
                    'y': y,
                    'confidence': conf
                })

        # Save annotated frame to output video
        out.write(annotated_frame)

    frame_num += 1

cap.release()
out.release()

# --- Save to CSV ---
df = pd.DataFrame(data)
df.to_csv(output_csv, index=False)
print(f"Keypoints saved to {output_csv}")
print(f"Video saved to {output_video_path}")

Code Block 3B

In [ ]:
# Load the model from Google Drive
model = joblib.load('/content/drive/My Drive/exercise_form/form_classifier_model.pkl')

# Load data
X = []
y = []

# --- Helper Function to Extract Features from a Keypoints CSV File ---
def extract_features_from_csv(csv_path):
    # Load keypoints CSV into a DataFrame
    df = pd.read_csv(csv_path)

    # Pivot the data: rows = frames, columns = (x, y) coordinates for each keypoint_id
    grouped = df.groupby(['frame', 'keypoint_id'])[['x', 'y']].mean().unstack()

    # --- Feature Extraction ---
    # Use statistical summaries:
    features = grouped.mean().tolist()  # Mean position of each keypoint (avg posture)
    features += grouped.std().tolist()  # Standard deviation (movement/instability)

    return features

features = extract_features_from_csv(output_csv)
if features:
    X.append(features)

# Convert to arrays
X = np.array(X)

# Evaluate
y_pred = model.predict(X)

if y_pred[0] == 0:
  print("That was great form!")
else:
  print("Form could use some work")